In [1]:
import _referAsMain
import sys; print(sys.version_info)
from datasets import load_dataset
import torch, random, time, math
from IPython.display import SVG, display
import paths_cfg

added '/home/holo/dev/PFE_LLM_art_generation' to import paths
sys.version_info(major=3, minor=10, micro=19, releaselevel='final', serial=0)


/home/holo/dev/PFE_LLM_art_generation/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


using default paths config


In [2]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device count:", torch.cuda.device_count())

Torch version: 2.9.1+cpu
CUDA available: False
CUDA version: None
Device count: 0


In [3]:
dataset = load_dataset("xingxm/SVGX-SFT-1M", split="train", data_files="SVGX_SFT_GEN_51k.json")

In [4]:
print(len(dataset))

514172


In [5]:
print(*dataset[0].items(), sep="\n")

('instruction', 'Generate an SVG illustration from the given description.')
('input', 'SVG illustration of person in suit levitating.')
('output', '<svg enable-background="new 0 0 128 128" viewBox="0 0 128 128" xmlns="http://www.w3.org/2000/svg" xmlns:xlink="http://www.w3.org/1999/xlink"><radialGradient id="a" cx="63.9995" cy="8.0164" gradientTransform="matrix(1 0 0 -.305 0 122.4395)" gradientUnits="userSpaceOnUse" r="15.2017"><stop offset=".1396" stop-color="#504f4f" stop-opacity=".8"/><stop offset=".8722" stop-color="#616161" stop-opacity="0"/></radialGradient><radialGradient id="b" cx="108.9787" cy="96.3683" gradientTransform="matrix(1 0 0 .5046 -39.7364 -17.1289)" gradientUnits="userSpaceOnUse" r="8.3693"><stop offset=".7275" stop-color="#6d4c41" stop-opacity="0"/><stop offset="1" stop-color="#6d4c41"/></radialGradient><radialGradient id="c" cx="100.7358" cy="97.6126" gradientTransform="matrix(-.9057 .4354 -.3144 -.6903 199.4155 53.7455)" gradientUnits="userSpaceOnUse" r="2.4733"><

In [6]:
import tokenizer_pfe.tokenizer_project as tokenizerLib
from tokenizer_pfe.tokenizer_project import SPECIAL_TOKENS

In [7]:
tokenizerPreTrained = tokenizerLib.Tokenizer.from_pretrained("gpt2")

In [8]:
tokenizerTrainedMedium = tokenizerLib.Tokenizer.train_from_iterator(
    dataset[: 5_000]["output"], vocab_size=2048, special_tokens=SPECIAL_TOKENS)

In [9]:
tokenizer = tokenizerTrainedMedium
vocab_size = tokenizer.get_vocab_size()
print(vocab_size)
display(sorted([(id, tk) for tk, id in tokenizer.tokenizer.get_vocab().items()], reverse=False))
OutStart_Token = "<|output_start|>"
OutEnd_Token = "<|output_end|>"
OutEnd_ids = tokenizer.encode(OutEnd_Token); assert len(OutEnd_ids) == 1
OutEnd_id = OutEnd_ids[0]; del OutEnd_ids

1101


[(0, '<|output_start|>'),
 (1, '<|output_end|>'),
 (2, '!'),
 (3, '"'),
 (4, '#'),
 (5, '$'),
 (6, '%'),
 (7, '&'),
 (8, "'"),
 (9, '('),
 (10, ')'),
 (11, '*'),
 (12, '+'),
 (13, ','),
 (14, '-'),
 (15, '.'),
 (16, '/'),
 (17, '0'),
 (18, '1'),
 (19, '2'),
 (20, '3'),
 (21, '4'),
 (22, '5'),
 (23, '6'),
 (24, '7'),
 (25, '8'),
 (26, '9'),
 (27, ':'),
 (28, ';'),
 (29, '<'),
 (30, '='),
 (31, '>'),
 (32, '?'),
 (33, '@'),
 (34, 'A'),
 (35, 'B'),
 (36, 'C'),
 (37, 'D'),
 (38, 'E'),
 (39, 'F'),
 (40, 'G'),
 (41, 'H'),
 (42, 'I'),
 (43, 'J'),
 (44, 'K'),
 (45, 'L'),
 (46, 'M'),
 (47, 'N'),
 (48, 'O'),
 (49, 'P'),
 (50, 'Q'),
 (51, 'R'),
 (52, 'S'),
 (53, 'T'),
 (54, 'U'),
 (55, 'V'),
 (56, 'W'),
 (57, 'X'),
 (58, 'Y'),
 (59, 'Z'),
 (60, '['),
 (61, '\\'),
 (62, ']'),
 (63, '^'),
 (64, '_'),
 (65, '`'),
 (66, 'a'),
 (67, 'b'),
 (68, 'c'),
 (69, 'd'),
 (70, 'e'),
 (71, 'f'),
 (72, 'g'),
 (73, 'h'),
 (74, 'i'),
 (75, 'j'),
 (76, 'k'),
 (77, 'l'),
 (78, 'm'),
 (79, 'n'),
 (80, 'o'),
 (81, 'p'

In [10]:
print(tokenizer.get_special_tokens())

['<|output_start|>', '<|output_end|>']


In [11]:
tokenizer.save(paths_cfg.TOKENIZER_SAVE_DIRECTORY.joinpath("test_tokenizer"))

saving the tokenizer to: /home/holo/dev/PFE_LLM_art_generation/tokenizer_save/test_tokenizer.json


In [12]:
loadedTokenizer = tokenizer.load(paths_cfg.TOKENIZER_SAVE_DIRECTORY.joinpath("test_tokenizer"))

loading the tokenizer from: /home/holo/dev/PFE_LLM_art_generation/tokenizer_save/test_tokenizer.json


In [13]:
vocab_size = loadedTokenizer.get_vocab_size()
print(vocab_size)

1101
